# 03 · The Verification Layer
**Grounded Conversational Chart Accessibility** — Section 3 of 6

**What this notebook does, in plain language**

1. **Reads each LLM answer** and pulls out what it actually claimed: a number, a name, or Yes/No. This works for short answers *and* full sentences.
2. **Compares the claim with the oracle's true answer** using plain code (no LLM judges anything):
   - numbers → tolerance check, names → RapidFuzz fuzzy matching, Yes/No → exact.
3. **Labels every answer** as correct or as one kind of hallucination.
4. **Re-prompts** the LLM for every flagged answer, giving it the verified value, then **verifies the new answer again**. If it's still wrong, a plain template answer is used instead, so the user never receives an unverified answer.
5. Exports a small **blind audit sheet** so *you* can check the verifier itself (its false-positive and false-negative rates).
6. Saves `outputs/verified.csv` with the labels and features for Section 4.

## Hallucination categories

| status | meaning | example |
|---|---|---|
| `correct` | matches the oracle | |
| `wrong_value` | wrong number, or wrong Yes/No | ratio 1.6 instead of 2.625 |
| `wrong_entity` | names the wrong category/year | "Ontario" instead of "British Columbia" |
| `wrong_answer_type` | number given when a name was asked (or vice versa) | "1001" instead of "Obesity" |
| `refusal` | claims the data isn't there (it always is) | "Not available" |
| `unparseable` / `no_answer` | nothing usable in the answer | |

Everything except `correct` counts as a **hallucination**.

**Matching rules (decided up front, and worth stating in the report):**
- **Lookups** must be within **0.5%** (the model only has to copy a value); **computed** answers (sum, avg, diff, ratio) within **5%**, ChartQA's standard tolerance.
- **Sentence answers:** a sentence often repeats several table values, so we take the number **closest** to the true value. This is a deliberately lenient "does it contain the right value?" rule. To stop it being fooled, numbers inside **table labels** (years, "25-34 age group"), **age ranges** ("65+") and the **question** are removed first.
- **Units:** "77.5 billion" is converted to the table's own unit (read from its column header, or from the question) before comparing. Number words followed by a unit ("three and a half billion", "two point five million") are read as numbers.
- **Names:** an answer names a table label only by a whole-word match ("Niger" is not found inside "Nigeria"). Short labels ("UK") and generic ones ("Total", "Other") must match with the same capitals, so "the total consumption" doesn't count as the row "Total".
- **Ratios** written `1:4` or `1 to 4` count as 0.25.
- **Trend** ("which year saw the biggest increase?"): either end of the change counts (2017 *or* 2018 for a 2017→2018 jump), since both are fair readings. ChartQA itself uses the end year.

## 3.1 Setup

In [2]:
# First time only (adds the cosine-similarity library):  py -3.12 -m pip install sentence-transformers
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from src import config
from src.data_utils import load_table
from src.verify import (verify_frame, cosine_similarities, reference_sentence,
                        template_answer, fmt_value)
from src.llm import (build_reprompt, ask, ResponseCache, QuotaExhausted,
                     ollama_ready, gemini_ready)

config.make_dirs()
config.set_seeds()
logging.basicConfig(filename=config.LOG_DIR / "03_verify.log", filemode="a",
                    level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
pd.set_option("display.max_colwidth", 90)

df = pd.read_csv(config.OUTPUT_DIR / "llm_answers.csv")
TABLES = {f: load_table(f) for f in df["table_file"].unique()}
print(f"{len(df)} answers loaded ({df['model'].nunique()} models x {df['style'].nunique()} styles)")

800 answers loaded (2 models x 2 styles)


## 3.2 Verify every answer

`verify_frame` runs the verifier on all answers. Besides the status it records:
- `claim`: what the verifier read from the answer,
- `num_deviation`: how far a number is from the truth (0.1 = 10% off),
- `fuzzy_score`: RapidFuzz similarity between the answer and the true name (0–100),
- `answer_len` (words) and `is_refusal`.

These become the classifier's features in Section 4.

In [3]:
v = pd.concat([df, verify_frame(df, TABLES)], axis=1)
v["hallucinated"] = v["status"] != "correct"

pd.crosstab([v["model"], v["style"]], v["status"], margins=True)

status                 correct  no_answer  refusal  unparseable  \
model        style                                                
gemini-flash sentence      197          0        0            0   
             short         194          0        1            0   
llama3.2-3b  sentence      139          0        3            1   
             short          82          1        1            3   
All                        612          1        5            4   

status                 wrong_answer_type  wrong_entity  wrong_value  All  
model        style                                                        
gemini-flash sentence                  0             1            2  200  
             short                     2             1            2  200  
llama3.2-3b  sentence                  1            14           42  200  
             short                    22            18           73  200  
All                                   25            34          119  800

**See it working.** A few real answers, what the verifier read from each, and its verdict. Change `random_state` to see others.

In [4]:
v.sample(12, random_state=1)[["model", "style", "qtype", "oracle_answer", "llm_answer", "claim", "status"]]

,model,style,qtype,oracle_answer,llm_answer,claim,status
8,llama3.2-3b,short,diff,0.2,2.07,2.07,wrong_value
510,gemini-flash,short,max,Fulham,Fulham,Fulham,correct
175,gemini-flash,sentence,compare,Yes,"Yes, the Female share in agriculture is higher than the Female share in industry.",Yes,correct
735,gemini-flash,sentence,max,Indonesia,The largest Chinese diaspora community lives in Indonesia with 10.72 million people.,Indonesia,correct
242,gemini-flash,short,min,9.29,9.29,9.29,correct
764,llama3.2-3b,short,trend,2020,2015,2015,wrong_entity
540,llama3.2-3b,short,max,Michael Kors,Michael Kors,Michael Kors,correct
430,gemini-flash,short,diff,6417.0,6417,6417.0,correct
84,llama3.2-3b,short,compare,0.9912,0.94,0.94,wrong_value
642,gemini-flash,short,min,Nepal,Nepal,Nepal,correct


## 3.3 Hallucination rate before verification

This is the headline "before" number: how often each model's raw answer is wrong.

In [5]:
def rate(frame, by):
    out = frame.groupby(by)["hallucinated"].agg(n="size", hallucinated="sum")
    out["rate_%"] = (100 * out["hallucinated"] / out["n"]).round(1)
    return out

before = rate(v, ["model", "style"])
before.to_csv(config.TABLE_OUT / "rate_by_model_style.csv")
before

n  hallucinated  rate_%
model        style                              
gemini-flash sentence  200             3     1.5
             short     200             6     3.0
llama3.2-3b  sentence  200            61    30.5
             short     200           118    59.0

In [6]:
# By question type (columns = model / style). Small n for avg, sum, compare, trend: read with care.
by_q = v.pivot_table(index="qtype", columns=["model", "style"], values="hallucinated", aggfunc="mean")
by_q = (100 * by_q).round(1)
by_q["n_questions"] = v.groupby("qtype")["qid"].nunique()
by_q.to_csv(config.TABLE_OUT / "rate_by_qtype.csv")
by_q

model   gemini-flash       llama3.2-3b       n_questions
style       sentence short    sentence short            
qtype                                                   
avg              0.0   0.0        50.0  83.3          12
compare          0.0   0.0        14.3  64.3          14
diff             4.4   4.4        55.6  86.7          45
lookup           0.0   0.0        22.2  20.0          45
max              0.0   6.7        13.3  60.0          45
min              4.5   4.5        36.4  68.2          22
sum              0.0   0.0         8.3  58.3          12
trend            0.0   0.0        60.0  40.0           5

In [7]:
# By chart type
by_c = (100 * v.pivot_table(index="chart_type", columns=["model", "style"],
                             values="hallucinated", aggfunc="mean")).round(1)
by_c["n_questions"] = v.groupby("chart_type")["qid"].nunique()
by_c.to_csv(config.TABLE_OUT / "rate_by_chart.csv")
by_c

model      gemini-flash       llama3.2-3b       n_questions
style          sentence short    sentence short            
chart_type                                                 
bar                 1.0   3.9        27.5  59.8         102
line                2.9   2.9        42.6  50.0          68
pie                 0.0   0.0        13.3  76.7          30

**Optional comparison with the standard ChartQA rule.** Re-checking with a flat 5% tolerance for everything (including lookups) shows how much the stricter lookup rule changes things.

In [8]:
import src.verify as V
_strict = V.LOOKUP_TOL
V.LOOKUP_TOL = config.REL_TOL                       # temporarily use 5% everywhere
relaxed = verify_frame(df, TABLES)["status"] != "correct"
V.LOOKUP_TOL = _strict                              # restore 0.5%
v["hallucinated_relaxed5"] = relaxed.values
(100 * v.groupby(["model", "style"])[["hallucinated", "hallucinated_relaxed5"]].mean()).round(1)

hallucinated  hallucinated_relaxed5
model        style                                        
gemini-flash sentence           1.5                    1.5
             short              3.0                    3.0
llama3.2-3b  sentence          30.5                   30.5
             short             59.0                   58.5

## 3.4 Cosine similarity (Sentence-Transformers)

We embed each answer and a reference sentence ("*question* The answer is *X*.") with `all-MiniLM-L6-v2` and measure cosine similarity (1 = same meaning).

**Why it is a feature, not the judge:** "*Database* has the lowest share" and "*Analytics* has the lowest share" are almost the same sentence, so their similarity is high even though one is wrong. The table below shows how well (or badly) cosine separates correct from hallucinated answers in *your* data, which is a useful finding for the report.

The first run downloads the model (~90 MB) and takes a minute or two on a CPU.

In [9]:
refs = [reference_sentence(q, fmt_value(o, k)) for q, o, k in
        zip(v["query"], v["oracle_answer"], v["answer_kind"])]
v["cosine_sim"] = cosine_similarities(v["llm_answer"].fillna(""), refs)

v.groupby(["model", "style", "hallucinated"])["cosine_sim"].mean().unstack().round(3)

c:\Users\navaneeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3052.88it/s]


hallucinated           False  True 
model        style                 
gemini-flash sentence  0.813  0.842
             short     0.377  0.200
llama3.2-3b  sentence  0.809  0.763
             short     0.373  0.220

## 3.5 Re-prompt flagged answers, then verify again

For every hallucinated answer, the LLM gets the same table and question, its own wrong answer, and the **verified value**, and is asked to restate the answer in the same style. The new answer is verified again:

- passes → it becomes the final answer (`final_source = "reprompt"`),
- fails → a template built from the oracle is used (`final_source = "template"`),
- answers that were correct first time are kept as they are (`final_source = "original"`).

Like Section 2, every call is cached; if the run stops, just rerun this cell. Make sure the Ollama app is running.

In [10]:
cache = ResponseCache()
ready = {m["name"]: (ollama_ready(m["model"]) if m["provider"] == "ollama" else gemini_ready())
         for m in config.MODELS}
model_cfg = {m["name"]: m for m in config.MODELS}

flagged = v[v["hallucinated"]]
print(f"{len(flagged)} answers to re-prompt:", flagged["model"].value_counts().to_dict())

reprompt_answers = {}
stopped = set()
for i, r in enumerate(flagged.itertuples(), 1):
    if not ready.get(r.model) or r.model in stopped:
        continue
    prompt = build_reprompt(r.query, r.table_file, r.chart_type, r.style,
                            r.llm_answer, fmt_value(r.oracle_answer, r.answer_kind))
    try:
        rec = ask(model_cfg[r.model], prompt, cache)
    except QuotaExhausted:
        print(f"{r.model}: free-tier limit reached; rerun later to finish.")
        stopped.add(r.model)
        continue
    reprompt_answers[r.Index] = rec["answer"]
    if i % 25 == 0:
        print(f"  {i}/{len(flagged)}")

v["reprompt_answer"] = pd.Series(reprompt_answers)
print(f"Re-prompted: {v['reprompt_answer'].notna().sum()} of {len(flagged)}")

188 answers to re-prompt: {'llama3.2-3b': 179, 'gemini-flash': 9}
  25/188
  50/188
  75/188
  100/188
  125/188
  150/188
  175/188
Re-prompted: 188 of 188


In [11]:
# Verify the re-prompted answers with exactly the same rules
rp = v[v["reprompt_answer"].notna()]
rp_check = verify_frame(rp, TABLES, answer_col="reprompt_answer")
v.loc[rp.index, "reprompt_status"] = rp_check["status"]

# Choose the final answer delivered to the user
def final(r):
    if not r.hallucinated:
        return r.llm_answer, "original"
    if r.reprompt_status == "correct":
        return r.reprompt_answer, "reprompt"
    return template_answer(r.oracle_answer, r.answer_kind), "template"

v[["final_answer", "final_source"]] = v.apply(lambda r: pd.Series(final(r)), axis=1)

# How often does each model accept the correction?
fixed = (v[v["hallucinated"]].groupby(["model", "style"])
         .agg(flagged=("hallucinated", "size"),
              reprompted=("reprompt_answer", lambda s: s.notna().sum()),
              fixed_by_reprompt=("reprompt_status", lambda s: (s == "correct").sum())))
fixed["fix_rate_%"] = (100 * fixed["fixed_by_reprompt"] / fixed["reprompted"]).round(1)
fixed

flagged  reprompted  fixed_by_reprompt  fix_rate_%
model        style                                                       
gemini-flash sentence        3           3                  3       100.0
             short           6           6                  6       100.0
llama3.2-3b  sentence       61          61                 57        93.4
             short         118         118                 99        83.9

In [12]:
v[v["final_source"] == "template"][["style", "qtype", "oracle_answer", "llm_answer", "reprompt_answer", "reprompt_status"]]


,style,qtype,oracle_answer,llm_answer,reprompt_answer,reprompt_status
1,sentence,lookup,13779.1,The total consumption of non-alcoholic beverages in the UK in 2018 was approximately 1...,The total consumption of non-alcoholic beverages in the UK in 2018 was approximately 1...,wrong_value
9,sentence,diff,0.2,"The difference between the highest and lowest exchange rate is 0.18, with the highest ...","The difference between the highest and lowest exchange rate is 0.18, with the highest ...",wrong_value
16,short,max,Obesity,1001,1001,wrong_answer_type
48,short,min,France,2,0,wrong_answer_type
105,sentence,lookup,55.0,The percentage of PET plastics collection rate in Italy is 42%.,The percentage of PET plastics collection rate in Italy is 42%.,wrong_value
152,short,min,Don't Know,6,6,wrong_answer_type
212,short,max,Malawi,49.57,49.57,wrong_answer_type
216,short,max,WhatsApp,51%,64,wrong_answer_type
288,short,min,9 years and younger,40 to 49 years,9,wrong_answer_type
341,sentence,lookup,6192.5,The estimated total health expenditure of the United States for 2028 is approximately ...,The estimated total health expenditure of the United States for 2028 is approximately ...,wrong_value


## 3.6 Before vs. after

- **Before** = how often the raw LLM answer is wrong (from 3.3).
- **After, as the verifier sees it** = 0% by design: every flagged answer is replaced by a re-verified or template answer.
- **After, honestly** = the hallucinations the verifier *missed*. That can only be measured against human judgement, which is what the audit in 3.7 is for. Don't report "0% after" without it.

In [13]:
summary = v.groupby(["model", "style"]).agg(
    answers=("hallucinated", "size"),
    before_rate_pct=("hallucinated", lambda s: round(100 * s.mean(), 1)),
    kept_original=("final_source", lambda s: (s == "original").sum()),
    fixed_by_reprompt=("final_source", lambda s: (s == "reprompt").sum()),
    template_fallback=("final_source", lambda s: (s == "template").sum()))
summary.to_csv(config.TABLE_OUT / "before_after.csv")
summary

answers  before_rate_pct  kept_original  \
model        style                                               
gemini-flash sentence      200              1.5            197   
             short         200              3.0            194   
llama3.2-3b  sentence      200             30.5            139   
             short         200             59.0             82   

                       fixed_by_reprompt  template_fallback  
model        style                                           
gemini-flash sentence                  3                  0  
             short                     6                  0  
llama3.2-3b  sentence                 57                  4  
             short                    99                 19

## 3.7 Audit the verifier (you do this part)

The verifier is code, and code can be wrong. To measure its **false-positive** rate (flagging a correct answer) and **false-negative** rate (missing a wrong one), a human must judge a sample **without seeing the verifier's verdict**.

The cell below writes `outputs/tables/audit_sheet.csv` with 60 answers the verifier flagged and 60 it passed, shuffled together, with the verdict hidden.

**How to fill it in (Excel is fine):**
1. For each row, compare `llm_answer` with `correct_answer`.
2. In `human_label` write **1** if the LLM answer gives the correct answer, or **0** if not. Use the **same rules as the verifier**, otherwise you'll be measuring a difference in rules rather than verifier mistakes:
   - **"What was X in 2019?"-style lookups must match exactly** (13,791.1 for 13779.1 is a **0**),
   - **calculated answers** (sum, average, difference, ratio) may be within about 5% (71.04 for 71.08 is a **1**),
   - a ratio written `1:4` or `1 to 4` equals 0.25,
   - a sentence counts if the answer it **states** is correct, even if other numbers appear in it,
   - for "which year saw the biggest increase", either year of that change counts.
3. Save it as **`audit_filled.csv`** in the same folder. In Excel, use *Save As → CSV UTF-8*.

Ideally two people label it independently and settle disagreements together; that makes the result much easier to defend.

In [14]:
AUDIT_PER_GROUP = 60
audit_src = pd.concat([
    v[v["hallucinated"]].sample(min(AUDIT_PER_GROUP, v["hallucinated"].sum()), random_state=config.SEED),
    v[~v["hallucinated"]].sample(AUDIT_PER_GROUP, random_state=config.SEED),
]).sample(frac=1, random_state=config.SEED)

audit = pd.DataFrame({
    "audit_id": range(1, len(audit_src) + 1),
    "row": audit_src.index,
    "question": audit_src["query"].values,
    "correct_answer": audit_src["label"].values,
    "llm_answer": audit_src["llm_answer"].values,
    "human_label": "",
})
audit_path = config.TABLE_OUT / "audit_sheet.csv"
if not (config.TABLE_OUT / "audit_filled.csv").exists():
    audit.to_csv(audit_path, index=False)
    print(f"Wrote {len(audit)} rows -> {audit_path}")
else:
    print("audit_filled.csv already exists; not overwriting the sheet.")

Wrote 120 rows -> D:\clg\IV\chart-hallucination\outputs\tables\audit_sheet.csv


**After filling it in**, run the next cell. Because flagged and passed answers were sampled separately, we estimate each group's error and scale it back up to all answers:
- **precision**: of answers the verifier flags, how many are truly wrong,
- **miss rate**: of answers it passes, how many are truly wrong,
- from these: false-positive rate, false-negative rate, recall, F1, and the **true residual hallucination rate after verification**.

In [17]:
filled_path = config.TABLE_OUT / "audit_filled.csv"
if not filled_path.exists():
    print("Fill in audit_sheet.csv and save it as audit_filled.csv, then rerun this cell.")
else:
    a = pd.read_csv(filled_path)
    a = a[a["human_label"].isin([0, 1, "0", "1"])].copy()
    a["truly_wrong"] = a["human_label"].astype(int) == 0
    a["flagged"] = v.loc[a["row"], "hallucinated"].values

    p_flag = a.loc[a["flagged"], "truly_wrong"].mean()        # precision
    p_miss = a.loc[~a["flagged"], "truly_wrong"].mean()       # miss rate among passed
    n_flag, n_pass = v["hallucinated"].sum(), (~v["hallucinated"]).sum()

    TP, FP = n_flag * p_flag, n_flag * (1 - p_flag)           # scale up to all answers
    FN, TN = n_pass * p_miss, n_pass * (1 - p_miss)
    recall = TP / (TP + FN)
    audit_metrics = pd.Series({
        "labelled_rows": len(a),
        "precision": p_flag,
        "recall": recall,
        "F1": 2 * p_flag * recall / (p_flag + recall),
        "false_positive_rate": FP / (FP + TN),
        "false_negative_rate": FN / (FN + TP),
        "residual_hallucination_after_%": 100 * FN / len(v),
    }).round(3)
    audit_metrics.to_csv(config.TABLE_OUT / "verifier_audit.csv")
    display(audit_metrics)
    print("\nDisagreements (verifier vs human):")
    display(a[a["flagged"] != a["truly_wrong"]][["question", "correct_answer", "llm_answer"]])

labelled_rows                     120.000
precision                           0.983
recall                              1.000
F1                                  0.992
false_positive_rate                 0.005
false_negative_rate                 0.000
residual_hallucination_after_%      0.000
dtype: float64


Disagreements (verifier vs human):


,question,correct_answer,llm_answer
102,What is the difference between the people of internet users between the age groups whi...,25,32% - 7%


## 3.8 Save for Section 4

In [18]:
keep = ["qid", "model", "style", "source", "table_file", "chart_type", "qtype", "answer_kind",
        "query", "label", "oracle_answer", "llm_answer", "claim", "status", "hallucinated",
        "hallucinated_relaxed5", "num_deviation", "fuzzy_score", "cosine_sim", "answer_len",
        "is_refusal", "latency_s", "reprompt_answer", "reprompt_status", "final_answer", "final_source"]
v[keep].to_csv(config.OUTPUT_DIR / "verified.csv", index=False)
print(f"Saved {len(v)} rows -> outputs/verified.csv")

Saved 800 rows -> outputs/verified.csv


## ✅ Checkpoint before Section 4

1. Read the 12 example rows in 3.2. Does each `claim` match what the answer actually says?
2. The before-rates in 3.3 and the re-prompt fix rates in 3.5 look sensible.
3. Fill in the audit sheet (3.7) and run the audit cell. **This is what lets you report the verifier's false-positive and false-negative rates honestly.**
4. `outputs/verified.csv` exists with 800 rows.